In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import BertTokenizer, BertModel, ViTModel, ViTImageProcessor

import numpy as np
from PIL import Image
from tqdm import tqdm

In [2]:
dataset = load_dataset("Zoe3324/flickr30k-pairs")

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/657 [00:00<?, ?B/s]

data/train-00000-of-00012.parquet:   0%|          | 0.00/97.0M [00:00<?, ?B/s]

data/train-00001-of-00012.parquet:   0%|          | 0.00/96.2M [00:00<?, ?B/s]

data/train-00002-of-00012.parquet:   0%|          | 0.00/101M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/145000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5070 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x7FD0FA6581A0>, 'caption': 'Two young guys with shaggy hair look at their hands while hanging out in the yard.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}


In [3]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [4]:
def collate_fn(batch):
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]

    # Create negative pairs by shifting captions by 1
    neg_texts = texts[1:] + texts[:1]

    all_texts = texts + neg_texts
    all_images = images + images  # each image paired with correct and wrong caption
    labels = [1.0] * len(texts) + [0.0] * len(texts)

    text_enc = tokenizer(
        all_texts,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors="pt"
    )

    image_enc = image_processor(
        images=all_images,
        return_tensors="pt"
    )

    return {
        "input_ids": text_enc["input_ids"],
        "attention_mask": text_enc["attention_mask"],
        "pixel_values": image_enc["pixel_values"],
        "labels": torch.tensor(labels, dtype=torch.float32),
    }

In [5]:
class CrossAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, q, k, v):
        Q = self.query(q)
        K = self.key(k)
        V = self.value(v)

        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, V)
        return out

In [6]:
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        dim = 768

        self.cross_attn = CrossAttention(dim)

        # MLP head: fused representation -> match score
        self.match_head = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        # Text features
        text_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state  # (B, T, D)

        # Image features
        image_out = self.image_encoder(
            pixel_values=pixel_values
        ).last_hidden_state  # (B, P, D)

        # Cross attention: text queries image
        fused = self.cross_attn(text_out, image_out, image_out)  # (B, T, D)

        # Mean pooling
        fused = fused.mean(dim=1)  # (B, D)

        # Match score
        logits = self.match_head(fused).squeeze(-1)  # (B,)
        return logits

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = MultiModalModel().to(device)

cuda


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def train(model, batch_size=32, epochs=5, lr=3e-5):
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        # Training
        model.train()
        total_train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask, pixel_values)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # Validation
        model.eval()
        total_val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                pixel_values = batch["pixel_values"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask, pixel_values)
                loss = criterion(logits, labels)
                total_val_loss += loss.item()

                preds = (torch.sigmoid(logits) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = correct / total

        print(f"Epoch {epoch+1}/{epochs} — Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

In [9]:
train(model, batch_size=32, epochs=1, lr=3e-5)

Epoch 1/1 [Val]: 100%|██████████| 159/159 [00:29<00:00,  5.46it/s]

Epoch 1/1 — Train Loss: 0.2108, Val Loss: 1.4430, Val Acc: 0.5985


In [10]:
torch.save(model.state_dict(), 'advanced_model_weights.pt')
# model.load_state_dict(torch.load('advanced_model_weights.pt', weights_only=True))

In [12]:
def eval_collate_fn(batch):
    """Collate without negatives — just return raw images and captions."""
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]
    return texts, images

In [13]:
def evaluate_retrieval(model, data, num_samples=200):
    """Evaluate image-to-text and text-to-image retrieval with Recall@k."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_fn)

    all_texts = []
    all_images = []
    for texts, images in loader:
        all_texts.extend(texts)
        all_images.extend(images)

    N = len(all_texts)
    score_matrix = torch.zeros(N, N)  # score_matrix[i][j] = match(image_i, caption_j)

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(N), desc="Scoring pairs"):
            # Pair image_i with all N captions
            img = all_images[i]
            image_enc = image_processor(images=[img] * N, return_tensors="pt")
            text_enc = tokenizer(all_texts, padding="max_length", truncation=True, max_length=32, return_tensors="pt")

            pixel_values = image_enc["pixel_values"].to(device)
            input_ids = text_enc["input_ids"].to(device)
            attention_mask = text_enc["attention_mask"].to(device)

            # Score in mini-batches to avoid OOM
            scores = []
            bs = 64
            for start in range(0, N, bs):
                end = min(start + bs, N)
                logits = model(input_ids[start:end], attention_mask[start:end], pixel_values[start:end])
                scores.append(torch.sigmoid(logits).cpu())

            score_matrix[i] = torch.cat(scores)

    # Image-to-text retrieval: for image_i, correct caption is i
    def recall_at_k(score_mat, k):
        correct = 0
        for i in range(score_mat.size(0)):
            topk = score_mat[i].topk(k).indices
            if i in topk:
                correct += 1
        return correct / score_mat.size(0)

    print()
    print(f"=== Image-to-Text Retrieval (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10):.4f}")

    # Text-to-image retrieval: transpose the matrix
    score_matrix_t = score_matrix.T
    print(f"=== Text-to-Image Retrieval (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix_t, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix_t, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix_t, 10):.4f}")

evaluate_retrieval(model, test_data, num_samples=200)

Scoring pairs: 100%|██████████| 200/200 [04:25<00:00,  1.33s/it]

=== Image-to-Text Retrieval (N=200) ===
  Recall@1:  0.1550
  Recall@5:  0.6950
  Recall@10: 0.9200
=== Text-to-Image Retrieval (N=200) ===
  Recall@1:  0.1450
  Recall@5:  0.7300
  Recall@10: 0.8800
